# Pre-entrega Módulo 5: Limpieza y Análisis Exploratorio de un Dataset Real
**Data Science I: Fundamentos de la Ciencia de Datos — Coderhouse**

Santiago Lliteras

Trabajo con el [Credit Risk Dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset) de Kaggle, el mismo de mis entregas anteriores. Cada fila es una solicitud de préstamo y la columna `loan_status` dice si terminó en default (1) o si se pagó con normalidad (0). La pregunta que guía mi proyecto es si se puede predecir el default a partir del perfil de la persona y de las condiciones del préstamo.

En esta entrega me concentro en lo que pide la consigna de este módulo: diagnosticar el dataset, limpiarlo (nulos, duplicados y tipos de datos) y responder preguntas de negocio con `groupby` y `pivot_table`. El CSV está en mi repositorio de GitHub, así que el notebook se puede ejecutar de principio a fin sin ningún archivo local.

## 1. Carga del dataset
Leo el CSV directamente desde mi repositorio de GitHub (el link "raw"), así cualquiera que abra el notebook lo puede correr sin tener que subir nada.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/santhor12/creditriskdataset/refs/heads/main/credit_risk_dataset.csv"
df = pd.read_csv(url)

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")

Filas: 32581  |  Columnas: 12


## 2. Diagnóstico inicial
Antes de tocar nada quiero entender en qué estado está el dataset. Voy a mirar las primeras filas, los tipos de datos, el resumen estadístico y cuántos valores faltan en cada columna.

In [2]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [4]:
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [5]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "porcentaje": porcentaje})

,nulos,porcentaje
person_age,0,0.00
person_income,0,0.00
person_home_ownership,0,0.00
person_emp_length,895,2.75
loan_intent,0,0.00
loan_grade,0,0.00
loan_amnt,0,0.00
loan_int_rate,3116,9.56
loan_status,0,0.00
loan_percent_income,0,0.00


**Qué veo:** el dataset tiene 32.581 filas y 12 columnas. Los tipos están bien: las columnas numéricas son `int64` o `float64`, y las de texto (`person_home_ownership`, `loan_intent`, `loan_grade` y `cb_person_default_on_file`) son `object`. No hay ningún número cargado como texto ni fechas para convertir.

El `describe()` confirma lo que ya había visto en la pre-entrega 3: `person_age` llega a 144 y `person_emp_length` a 123, valores que no pueden ser reales. Y `isnull().sum()` muestra faltantes en solo dos columnas: `person_emp_length` (895 filas, un 2,75 %) y `loan_int_rate` (3.116 filas, un 9,56 %).

## 3. Limpieza de datos

### 3.1 Valores imposibles y duplicados
Esto ya lo analicé en detalle en la pre-entrega 3, así que acá aplico el mismo criterio pero de forma resumida. Lo hago para que este notebook parta del CSV original y corra completo, sin depender de otra entrega. Quito las edades de más de 100 años, las antigüedades laborales de más de 60 y los duplicados exactos.

Dos detalles de código:
- En el filtro de antigüedad agrego `| isnull()` para que los nulos pasen el filtro. Los voy a tratar más abajo y no quiero borrarlos por accidente.
- Termino con `reset_index(drop=True)`. Al borrar filas el índice queda con huecos (0, 1, 3, 4...) y esto lo vuelve a numerar sin huecos. El `drop=True` evita que el índice viejo se guarde como una columna nueva.

In [6]:
print(f"Filas iniciales: {len(df)}")
print(f"Edades mayores a 100: {(df['person_age'] > 100).sum()}")
print(f"Antigüedades laborales mayores a 60: {(df['person_emp_length'] > 60).sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")

df = df[df["person_age"] <= 100]
df = df[(df["person_emp_length"] <= 60) | (df["person_emp_length"].isnull())]
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print(f"\nFilas luego de limpiar: {len(df)}")

Filas iniciales: 32581
Edades mayores a 100: 5
Antigüedades laborales mayores a 60: 2


Filas duplicadas: 165

Filas luego de limpiar: 32409


Se fueron 172 filas en total (5 por edad, 2 por antigüedad y 165 duplicadas) y quedan 32.409.

### 3.2 Nulos en `person_emp_length` (antigüedad laboral)
Acá tengo que elegir con qué rellenar. La consigna habla de media, moda o eliminar. Comparo la media con la mediana y miro el sesgo (`skew`), que me dice si hay valores muy altos que estiren la media. Además miro si las filas que no tienen este dato se comportan distinto en cuanto a default.

In [7]:
antiguedad = df["person_emp_length"]
print(f"Media:   {antiguedad.mean():.2f}")
print(f"Mediana: {antiguedad.median():.1f}")
print(f"Moda:    {antiguedad.mode()[0]:.1f}")
print(f"Sesgo (skew): {antiguedad.skew():.2f}")
print(f"Nulos: {antiguedad.isnull().sum()}")

print("\nTasa de default según si falta o no la antigüedad:")
falta = antiguedad.isnull().rename("falta_antiguedad")
print(df.groupby(falta)["loan_status"].agg(filas="size", tasa_default="mean").round(3))

Media:   4.78
Mediana: 4.0
Moda:    0.0
Sesgo (skew): 1.25
Nulos: 887

Tasa de default según si falta o no la antigüedad:
                  filas  tasa_default
falta_antiguedad                     
False             31522         0.216
True                887         0.317


La media da 4,78 y la mediana 4,0, con un sesgo de 1,25. Es positivo, o sea que hay una cola hacia antigüedades altas que estira la media. Por eso uso la **mediana**, que representa mejor a una persona típica. La moda no me sirve: es 0 y no describe al grupo.

Algo que me llamó la atención: donde falta la antigüedad el default es del 31,7 %, contra un 21,6 % cuando el dato sí está. Para 887 filas es una diferencia grande, así que es probable que el hecho de que falte ya tenga información. Al rellenar con la mediana esa señal desaparece. Igual lo hago, porque borrar 887 filas me parece peor, pero lo dejo anotado como limitación.

In [8]:
mediana_antiguedad = df["person_emp_length"].median()
df["person_emp_length"] = df["person_emp_length"].fillna(mediana_antiguedad)

print(f"Nulos restantes en person_emp_length: {df['person_emp_length'].isnull().sum()}")

Nulos restantes en person_emp_length: 0


### 3.3 Nulos en `loan_int_rate` (tasa de interés)
Acá primero me pregunto si la tasa depende de otra columna. Sospecho que del grado del préstamo (`loan_grade`), porque debería ser más alta cuanto peor es el grado. Lo compruebo con un `groupby`, y miro también si estos nulos esconden algo en cuanto a default.

In [9]:
print(f"Media general de la tasa: {df['loan_int_rate'].mean():.2f}")
print(f"Nulos: {df['loan_int_rate'].isnull().sum()}\n")

print("Tasa de interés por grado (sin contar los nulos):")
print(df.groupby("loan_grade")["loan_int_rate"].agg(["count", "mean", "median"]).round(2))

print("\nTasa de default según si falta o no la tasa:")
falta = df["loan_int_rate"].isnull().rename("falta_tasa")
print(df.groupby(falta)["loan_status"].agg(filas="size", tasa_default="mean").round(3))

Media general de la tasa: 11.02
Nulos: 3094

Tasa de interés por grado (sin contar los nulos):
            count   mean  median
loan_grade                      
A            9712   7.33    7.49
B            9335  11.00   10.99
C            5807  13.46   13.48
D            3308  15.36   15.31
E             880  17.01   16.82
F             214  18.61   18.54
G              59  20.25   20.16

Tasa de default según si falta o no la tasa:
            filas  tasa_default
falta_tasa                     
False       29315         0.220
True         3094         0.207


Se confirma: la tasa media sube de 7,33 % en el grado A a 20,25 % en el G. Si rellenara con la media general (11,02 %) le estaría poniendo esa tasa a un préstamo A, que normalmente ronda el 7 %, y también a uno G, que ronda el 20 %. Eso metería datos que no se parecen a la realidad. Por eso relleno cada nulo con la **media de su propio grado**.

En este caso los nulos no parecen esconder nada: el default es 20,7 % cuando falta la tasa y 22,0 % cuando está.

Cómo funciona el código: `transform("mean")` devuelve una serie del mismo largo que el dataframe, donde cada fila trae la media de su grado. Después `fillna` usa esa serie solo en las filas donde hay un nulo y deja intactos los valores que ya estaban.

In [10]:
tasa_media_por_grado = df.groupby("loan_grade")["loan_int_rate"].transform("mean")
df["loan_int_rate"] = df["loan_int_rate"].fillna(tasa_media_por_grado)

print(f"Nulos restantes en loan_int_rate: {df['loan_int_rate'].isnull().sum()}")

Nulos restantes en loan_int_rate: 0


### 3.4 Cambio de tipo de dato en `person_emp_length`
Esta columna era `float64` solo porque tenía nulos (un entero normal de pandas no puede guardar un valor faltante). Ahora que ya no hay nulos tiene más sentido que sea entera, porque los años de antigüedad son números enteros. Antes de convertir compruebo que no se pierda ningún decimal.

In [11]:
print("¿Todos los valores son enteros?", (df["person_emp_length"] % 1 == 0).all())
print("Tipo antes:", df["person_emp_length"].dtype)

df["person_emp_length"] = df["person_emp_length"].astype(int)

print("Tipo después:", df["person_emp_length"].dtype)

¿Todos los valores son enteros? True
Tipo antes: float64
Tipo después: int64


### 3.5 Chequeo final de la limpieza

In [12]:
print(f"Nulos en todo el dataset: {df.isnull().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"Forma final: {df.shape[0]} filas x {df.shape[1]} columnas\n")
df.info()

Nulos en todo el dataset: 0


Filas duplicadas: 0
Forma final: 32409 filas x 12 columnas



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32409 entries, 0 to 32408
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32409 non-null  int64  
 1   person_income               32409 non-null  int64  
 2   person_home_ownership       32409 non-null  object 
 3   person_emp_length           32409 non-null  int64  
 4   loan_intent                 32409 non-null  object 
 5   loan_grade                  32409 non-null  object 
 6   loan_amnt                   32409 non-null  int64  
 7   loan_int_rate               32409 non-null  float64
 8   loan_status                 32409 non-null  int64  
 9   loan_percent_income         32409 non-null  float64
 10  cb_person_default_on_file   32409 non-null  object 
 11  cb_person_cred_hist_length  32409 non-null  int64  
dtypes: float64(2), int64(6), object(4)
memory usage: 3.0+ MB


Quedó sin nulos, sin duplicados, con 32.409 filas y las 12 columnas originales. Para la limpieza usé `drop_duplicates`, `fillna`, `astype` y `reset_index`, además de filtros booleanos para los valores imposibles.

## 4. Agregaciones de negocio
`loan_status` vale 1 si hubo default y 0 si no. Por eso, el promedio de esa columna en un grupo es la proporción de préstamos que cayeron en default. Le llamo **tasa de default** y la muestro en porcentaje. En cada tabla agrego también la cantidad de préstamos del grupo, para no interpretar porcentajes de grupos muy chicos como si fueran sólidos.

### 4.1 ¿Cómo cambia el default según el grado del préstamo?

In [13]:
por_grado = df.groupby("loan_grade").agg(
    prestamos=("loan_status", "size"),
    tasa_default_pct=("loan_status", "mean"),
    monto_medio=("loan_amnt", "mean"),
)
por_grado["tasa_default_pct"] = (por_grado["tasa_default_pct"] * 100).round(1)
por_grado["monto_medio"] = por_grado["monto_medio"].round(0)
por_grado

,prestamos,tasa_default_pct,monto_medio
loan_grade,,,
A,10702,10.0,8545.0
B,10384,16.3,9991.0
C,6436,20.8,9221.0
D,3619,59.0,10849.0
E,963,64.5,12920.0
F,241,70.5,14717.0
G,64,98.4,17196.0


El default sube con cada grado, pero no de forma pareja: de A a C pasa de 10,0 % a 20,8 %, y entre C y D salta a 59,0 %. Del grado D para arriba más de la mitad de los préstamos terminan en default, y el G llega a 98,4 %. Ese último número hay que tomarlo con cuidado porque son solo 64 préstamos.

### 4.2 ¿Y según el tipo de vivienda del solicitante?

In [14]:
por_vivienda = df.groupby("person_home_ownership").agg(
    prestamos=("loan_status", "size"),
    tasa_default_pct=("loan_status", "mean"),
    ingreso_mediano=("person_income", "median"),
)
por_vivienda["tasa_default_pct"] = (por_vivienda["tasa_default_pct"] * 100).round(1)
por_vivienda.sort_values("tasa_default_pct", ascending=False)

,prestamos,tasa_default_pct,ingreso_mediano
person_home_ownership,,,
RENT,16374,31.6,48000.0
OTHER,106,31.1,59500.0
MORTGAGE,13366,12.6,69000.0
OWN,2563,7.5,47000.0


Alquilar es lo más riesgoso: 31,6 % de default, contra 12,6 % con hipoteca y 7,5 % con vivienda propia. `OTHER` tiene 31,1 %, pero son solo 106 casos.

Algo curioso de la columna de ingreso mediano: quienes tienen vivienda propia son los de menor ingreso (47.000) y aun así tienen el menor default. Así que el tipo de vivienda no es simplemente "quien gana más, paga mejor".

### 4.3 ¿Y según para qué se pide el préstamo?

In [15]:
por_destino = df.groupby("loan_intent").agg(
    prestamos=("loan_status", "size"),
    tasa_default_pct=("loan_status", "mean"),
    monto_medio=("loan_amnt", "mean"),
)
por_destino["tasa_default_pct"] = (por_destino["tasa_default_pct"] * 100).round(1)
por_destino["monto_medio"] = por_destino["monto_medio"].round(0)
por_destino.sort_values("tasa_default_pct", ascending=False)

,prestamos,tasa_default_pct,monto_medio
loan_intent,,,
DEBTCONSOLIDATION,5189,28.7,9605.0
MEDICAL,6042,26.8,9272.0
HOMEIMPROVEMENT,3594,26.2,10362.0
PERSONAL,5496,19.9,9576.0
EDUCATION,6409,17.3,9479.0
VENTURE,5679,14.9,9579.0


Los destinos más riesgosos son consolidar deudas (`DEBTCONSOLIDATION`, 28,7 %), gastos médicos (26,8 %) y mejoras del hogar (26,2 %). Los más seguros son emprendimiento (`VENTURE`, 14,9 %) y educación (17,3 %). Todos los grupos tienen entre 3.594 y 6.409 préstamos, así que acá no hay problema de grupos chicos.

### 4.4 Grado y vivienda a la vez (`pivot_table`)
Cruzo las dos variables que más separaron el riesgo para ver si el tipo de vivienda sigue importando dentro del mismo grado, o si era solo otra manera de ver el grado. Primero la tasa de default y después la cantidad de préstamos de cada celda.

In [16]:
pivot_default = df.pivot_table(
    index="loan_grade",
    columns="person_home_ownership",
    values="loan_status",
    aggfunc="mean",
)
(pivot_default * 100).round(1)

person_home_ownership,MORTGAGE,OTHER,OWN,RENT
loan_grade,,,,
A,4.4,11.5,7.1,17.5
B,8.0,15.2,4.3,24.1
C,14.3,35.3,6.3,26.6
D,45.6,55.0,6.4,73.8
E,47.6,75.0,54.4,75.5
F,62.9,100.0,53.3,78.0
G,100.0,NaN,100.0,96.4


In [17]:
pivot_cantidad = df.pivot_table(
    index="loan_grade",
    columns="person_home_ownership",
    values="loan_status",
    aggfunc="count",
    fill_value=0,
)
pivot_cantidad

person_home_ownership,MORTGAGE,OTHER,OWN,RENT
loan_grade,,,,
A,5417,26,917,4342
B,4025,33,797,5529
C,2248,17,492,3679
D,1210,20,280,2109
E,338,8,57,560
F,97,2,15,127
G,31,0,5,28


Aun dentro del mismo grado, alquilar sigue siendo más riesgoso que tener hipoteca: en el grado A es 17,5 % contra 4,4 %, y en el B 24,1 % contra 8,0 %. Así que el tipo de vivienda aporta información propia y no es solo el grado dicho de otra forma.

Hay que tener cuidado con las celdas chicas. El grado G con vivienda propia da 100 %, pero son 5 préstamos, y el grado G con `OTHER` ni existe (por eso aparece `NaN`).

### 4.5 ¿En qué se diferencian los que cayeron en default?
Comparo las medianas de cuatro variables según `loan_status`. Uso la mediana y no la media porque el ingreso tiene valores extremos que distorsionarían el promedio.

In [18]:
df.groupby("loan_status")[["loan_percent_income", "loan_int_rate", "person_income", "loan_amnt"]].median().round(2)

,loan_percent_income,loan_int_rate,person_income,loan_amnt
loan_status,,,,
0,0.13,10.65,60000.0,8000.0
1,0.24,13.46,41655.0,9600.0


Los que cayeron en default (`loan_status` = 1) piden préstamos que pesan más sobre su ingreso (mediana de `loan_percent_income` de 0,24 contra 0,13), con una tasa más alta (13,46 % contra 10,65 %) y con un ingreso mediano menor (41.655 contra 60.000). El monto mediano también es algo mayor (9.600 contra 8.000), aunque es la diferencia más chica de las cuatro.

## 5. Reflexión

**¿Qué problemas encontré?** Dos columnas con nulos (`person_emp_length` y `loan_int_rate`), valores imposibles en edad y antigüedad laboral, y 165 filas duplicadas. Los tipos de datos estaban bien, salvo que `person_emp_length` era decimal por culpa de los nulos.

**¿Qué decidí y por qué?** Rellené la antigüedad laboral con la mediana (4 años) porque la media estaba estirada por los casos de muchos años. Rellené la tasa de interés con la media de cada grado, porque la tasa depende mucho del grado y una media general le habría puesto una tasa poco realista a casi todos los préstamos.

**¿Qué me dicen las agrupaciones?** El grado es lo que más separa el riesgo (de 10,0 % en el A a 98,4 % en el G, con un salto fuerte entre C y D). El tipo de vivienda también aporta, incluso dentro del mismo grado. Los que caen en default piden préstamos que pesan más sobre su ingreso y tienen tasas más altas.

**Limitaciones que quiero dejar anotadas:**
- Rellené 887 antigüedades y 3.094 tasas (alrededor del 9,5 % de la columna de tasas): esos valores no son observaciones reales. Y en el caso de la antigüedad, los nulos tenían más default que el resto, información que se pierde al imputar.
- Algunos grupos son muy chicos (grado G con 64 préstamos, vivienda `OTHER` con 106), por lo que sus porcentajes son poco confiables.
- Cuando llegue al modelado voy a tener que revisar esta imputación: lo habitual es calcular las medianas y medias solo con los datos de entrenamiento, para no usar información del conjunto de prueba.

El dataset queda limpio y listo para la siguiente entrega, la de visualizaciones, que ya no tiene que lidiar con nulos.